<a href="https://colab.research.google.com/github/r-yv/PromptEng/blob/main/reACT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Tools: Claude
# Promt: I (as the customer) want to ask a question about shipping delivery. The customer should be able to write something.
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║         ReACT Framework: Prompt Chaining for Customer Support AI             ║
║         Standard Library Only — Google Colab Compatible                      ║
╚══════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STAGE 1: REASON AND PLAN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PROBLEM ANALYSIS
─────────────────
Simulate a multi-stage customer support prompt chain entirely in Python.
Each stage transforms a shared state object, mimicking how an LLM pipeline
would operate — classify intent, enrich with data, plan resolution, draft a
reply, and QA-check it — without any external API calls.

Functional Requirements:
  FR-1  Accept a free-text customer message as input.
  FR-2  Classify intent: category, subcategory, sentiment, priority.
  FR-3  Enrich context: fetch mock order records and policy snippets.
  FR-4  Determine the best resolution action with a written plan.
  FR-5  Draft a personalised, empathetic customer-facing reply.
  FR-6  QA-check the draft against rubric criteria; auto-fix minor gaps.
  FR-7  Display a structured, stage-by-stage execution log.
  FR-8  Support both batch-demo mode and interactive REPL mode.

SYSTEM DESIGN
──────────────
  CustomerContext   — frozen dataclass carrying all state between stages.
  OrderDB           — dict-based mock database.
  PolicyStore       — dict-based policy knowledge base.
  Classifier        — keyword + regex engine (simulates Stage-1 LLM call).
  ContextGatherer   — fetches order record + policies + extracts facts.
  ResolutionPlanner — decision-tree engine (simulates Stage-3 LLM call).
  ReplyDrafter      — template engine (simulates Stage-4 LLM call).
  QAChecker         — rubric evaluator (simulates Stage-5 LLM call).
  Chain             — orchestrator wiring all five stages together.
  Logger            — structured pretty-printer for stage logs.

CONSTRAINT IDENTIFICATION
──────────────────────────
  Permitted : Python standard library only.
  Used      : re, dataclasses, textwrap, sys, copy, datetime, enum

EDGE CASE STRATEGY
───────────────────
  EC-1  Empty / whitespace-only message   → raise ValueError before chain runs.
  EC-2  Order ID present but not in DB    → order_record = None; chain
                                            adapts resolution to "ask customer".
  EC-3  No order ID detected              → graceful degradation to info-only
                                            or clarification flow.
  EC-4  Unknown category                  → defaults to "general_inquiry".
  EC-5  QA fails on multiple criteria     → auto-patch where possible,
                                            flag remaining issues in output.
  EC-6  Keyboard interrupt in interactive → caught cleanly; prints farewell.
  EC-7  Non-string input                  → coerced to str via str().

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STAGE 2: GENERATE CODE (ACT)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

# ── Standard-library imports only ─────────────────────────────────────────────
import re
import sys
import copy
import textwrap
from dataclasses import dataclass, field
from typing      import Optional, List, Dict
from enum        import Enum
from datetime    import datetime


# ══════════════════════════════════════════════════════════════════════════════
# CONSTANTS & ENUMERATIONS
# ══════════════════════════════════════════════════════════════════════════════

class Category(str, Enum):
    ORDER_STATUS    = "order_status"
    RETURNS_REFUNDS = "returns_refunds"
    PRODUCT_ISSUE   = "product_issue"
    CANCELLATION    = "cancellation"
    PAYMENT         = "payment"
    SHIPPING_INQUIRY= "shipping_inquiry"
    GENERAL         = "general_inquiry"

class Sentiment(str, Enum):
    POSITIVE   = "positive"
    NEUTRAL    = "neutral"
    FRUSTRATED = "frustrated"
    ANGRY      = "angry"

class Priority(str, Enum):
    LOW    = "low"
    MEDIUM = "medium"
    HIGH   = "high"
    URGENT = "urgent"

class ResolutionType(str, Enum):
    PROVIDE_TRACKING      = "provide_tracking"
    PROCESSING_UPDATE     = "processing_update"
    CONFIRM_DELIVERED     = "confirm_delivered"
    ASK_ORDER_ID          = "ask_for_order_id"
    INITIATE_RETURN       = "initiate_return"
    REPLACEMENT_OR_REFUND = "send_replacement_or_refund"
    CANCEL_ORDER          = "cancel_order"
    CANNOT_CANCEL         = "cannot_cancel"
    ESCALATE_BILLING      = "escalate_billing"
    PROVIDE_INFO          = "provide_info"


# ══════════════════════════════════════════════════════════════════════════════
# DATA LAYER  —  mock order DB and policy store
# ══════════════════════════════════════════════════════════════════════════════

ORDER_DB: Dict[str, dict] = {
    "ORD-1001": {
        "status": "shipped",  "item": "Wireless Noise-Cancelling Headphones",
        "tracking": "UPS-4839201XZ", "carrier": "UPS",
        "shipped_date": "2026-02-24", "est_delivery": "2026-03-01",
        "total": "$129.99",
    },
    "ORD-1002": {
        "status": "processing", "item": "Mechanical Keyboard (TKL, Blue Switches)",
        "tracking": None, "carrier": None,
        "shipped_date": None, "est_delivery": "2026-03-04",
        "total": "$89.99",
    },
    "ORD-1003": {
        "status": "delivered", "item": "4K Webcam",
        "tracking": "FDX-00291847ZZ", "carrier": "FedEx",
        "shipped_date": "2026-02-18", "est_delivery": "2026-02-21",
        "total": "$74.99",
    },
    "ORD-1004": {
        "status": "cancelled", "item": "USB-C Hub (7-in-1)",
        "tracking": None, "carrier": None,
        "shipped_date": None, "est_delivery": None,
        "total": "$39.99",
    },
}

POLICY_STORE: Dict[str, str] = {
    "returns": (
        "We offer a 30-day hassle-free return policy. Items must be unused "
        "and in original packaging. Return shipping is free for defective items."
    ),
    "refunds": (
        "Refunds are issued within 3-5 business days after we receive the "
        "returned item. The refund appears on your original payment method."
    ),
    "shipping": (
        "Standard shipping (5-7 days) is free on orders over $50. "
        "Express 2-day shipping is available for $9.99."
    ),
    "warranty": (
        "All electronics include a 1-year manufacturer warranty covering "
        "defects in materials and workmanship."
    ),
    "cancellation": (
        "Orders can be cancelled within 1 hour of placement. After that "
        "the order enters processing and cannot be cancelled."
    ),
    "damaged_items": (
        "If you received a damaged item, please contact us within 48 hours "
        "with a photo. We will send a free replacement or issue a full refund."
    ),
}

CATEGORY_POLICY_MAP: Dict[str, List[str]] = {
    Category.ORDER_STATUS:     ["shipping"],
    Category.RETURNS_REFUNDS:  ["returns", "refunds"],
    Category.PRODUCT_ISSUE:    ["damaged_items", "returns", "warranty"],
    Category.CANCELLATION:     ["cancellation"],
    Category.PAYMENT:          ["refunds"],
    Category.SHIPPING_INQUIRY: ["shipping"],
}


# ══════════════════════════════════════════════════════════════════════════════
# SHARED STATE  —  CustomerContext dataclass
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class CustomerContext:
    """
    Single mutable state object that travels through every chain stage.
    Each stage reads fields it needs and writes new fields as outputs.
    """
    # ── Input ──────────────────────────────────────────────────────────────
    raw_message:          str

    # ── Stage 1 outputs  (classification) ─────────────────────────────────
    category:             Optional[str]  = None
    subcategory:          Optional[str]  = None
    sentiment:            Optional[str]  = None
    priority:             Optional[str]  = None
    detected_order_id:    Optional[str]  = None
    detected_name:        Optional[str]  = None

    # ── Stage 2 outputs  (context enrichment) ─────────────────────────────
    order_record:         Optional[dict] = None
    policy_snippets:      List[str]      = field(default_factory=list)
    extracted_facts:      List[str]      = field(default_factory=list)

    # ── Stage 3 outputs  (resolution planning) ────────────────────────────
    resolution_type:      Optional[str]  = None
    resolution_plan:      Optional[str]  = None
    needs_clarification:  bool           = False
    clarifying_questions: List[str]      = field(default_factory=list)

    # ── Stage 4 outputs  (draft reply) ────────────────────────────────────
    draft_reply:          Optional[str]  = None

    # ── Stage 5 outputs  (QA) ─────────────────────────────────────────────
    qa_passed:            bool           = False
    qa_issues:            List[str]      = field(default_factory=list)
    final_reply:          Optional[str]  = None

    # ── Execution metadata ─────────────────────────────────────────────────
    stage_timings:        Dict[str, str] = field(default_factory=dict)
    errors:               List[str]      = field(default_factory=list)

    def __post_init__(self):
        # EC-7: coerce non-string input
        self.raw_message = str(self.raw_message).strip()
        # EC-1: reject empty messages
        if not self.raw_message:
            raise ValueError("raw_message must not be empty or whitespace-only.")


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 1  —  CLASSIFIER
# ══════════════════════════════════════════════════════════════════════════════

class Classifier:
    """
    Simulates an LLM classification call using keyword matching + regex.
    Determines: category, subcategory, sentiment, priority, order_id, name.
    """

    # Keyword sets ordered by specificity (most specific checked first)
    _CATEGORY_RULES = [
        (Category.CANCELLATION,     ["cancel", "cancellation"]),
        (Category.PRODUCT_ISSUE,    ["broken", "damaged", "defective", "not working",
                                     "cracked", "wrong item", "doesn't work",
                                     "doesnt work", "faulty", "malfunction"]),
        (Category.RETURNS_REFUNDS,  ["return", "send back", "refund", "money back",
                                     "exchange", "replace it"]),
        (Category.SHIPPING_INQUIRY, ["free shipping", "shipping cost", "delivery fee",
                                     "how much does", "express delivery",
                                     "shipping policy", "how long does shipping"]),
        (Category.PAYMENT,          ["pay", "charge", "billed", "invoice", "payment",
                                     "overcharged", "double charged"]),
        (Category.ORDER_STATUS,     ["where is", "tracking", "hasn't arrived",
                                     "not arrived", "still waiting", "delayed",
                                     "missing", "not received", "when will",
                                     "ship", "delivery", "locate my"]),
    ]

    _ANGRY_WORDS     = {"unacceptable", "furious", "disgusted", "terrible",
                        "worst", "ridiculous", "outraged", "scam", "fraud",
                        "lawsuit", "disgusting", "appalling"}
    _FRUSTRATED_WORDS= {"frustrated", "annoyed", "disappointed", "still waiting",
                        "again", "still haven", "can't believe", "weeks",
                        "keep waiting", "no update", "useless"}
    _POSITIVE_WORDS  = {"thank", "great", "happy", "love", "appreciate", "awesome",
                        "excellent", "wonderful", "fantastic", "pleased"}

    # EC-4: default fallback category
    _DEFAULT_CATEGORY    = Category.GENERAL
    _DEFAULT_SUBCATEGORY = "general question"

    @staticmethod
    def _extract_order_id(msg: str) -> Optional[str]:
        """Extract and normalise order IDs like ORD1001, ORD-1001."""
        pattern = re.compile(r'\bord-?\d{3,6}\b', re.IGNORECASE)
        match = pattern.search(msg)
        if match:
            digits = re.sub(r'[^0-9]', '', match.group(0))
            return f"ORD-{digits}"
        return None

    @staticmethod
    def _extract_name(msg: str) -> Optional[str]:
        """Simple heuristic: 'my name is <Name>'."""
        pattern = re.compile(r'\bmy name is ([A-Z][a-z]+)\b', re.IGNORECASE)
        match = pattern.search(msg)
        return match.group(1).title() if match else None

    def classify(self, ctx: CustomerContext) -> CustomerContext:
        msg_lower = ctx.raw_message.lower()

        # ── Order ID & Name ──────────────────────────────────────────────
        ctx.detected_order_id = self._extract_order_id(ctx.raw_message)
        ctx.detected_name     = self._extract_name(ctx.raw_message)

        # ── Category (first matching rule wins) ──────────────────────────
        ctx.category    = self._DEFAULT_CATEGORY
        ctx.subcategory = self._DEFAULT_SUBCATEGORY
        for cat, keywords in self._CATEGORY_RULES:
            if any(kw in msg_lower for kw in keywords):
                ctx.category    = cat
                ctx.subcategory = cat.replace("_", " ")
                break

        # ── Sentiment ────────────────────────────────────────────────────
        words = set(re.findall(r'\w+', msg_lower))
        if self._ANGRY_WORDS & words or "!!!" in ctx.raw_message:
            ctx.sentiment = Sentiment.ANGRY
        elif self._FRUSTRATED_WORDS & words:
            ctx.sentiment = Sentiment.FRUSTRATED
        elif self._POSITIVE_WORDS & words:
            ctx.sentiment = Sentiment.POSITIVE
        else:
            ctx.sentiment = Sentiment.NEUTRAL

        # ── Priority (sentiment-driven with product-issue override) ──────
        priority_map = {
            Sentiment.ANGRY:     Priority.URGENT,
            Sentiment.FRUSTRATED:Priority.HIGH,
            Sentiment.NEUTRAL:   Priority.MEDIUM,
            Sentiment.POSITIVE:  Priority.LOW,
        }
        ctx.priority = priority_map.get(ctx.sentiment, Priority.MEDIUM)
        if ctx.category == Category.PRODUCT_ISSUE:
            ctx.priority = Priority.HIGH   # always escalate product defects

        return ctx


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 2  —  CONTEXT GATHERER
# ══════════════════════════════════════════════════════════════════════════════

class ContextGatherer:
    """
    Simulates an LLM tool-use / RAG step.
    Fetches the order record, selects relevant policy snippets,
    and extracts key facts from the raw message.
    """

    _FACT_PATTERNS = [
        (r'\b(week|weeks)\b',              "Customer has been waiting multiple weeks"),
        (r'\b(photo|picture|image)\b',     "Customer has photographic evidence"),
        (r'\balready.{0,20}contact\b',     "Customer has previously contacted support"),
        (r'\bgift\b',                      "Item may be a gift — urgency elevated"),
        (r'\bwork(s)?\b.*\bno(t)?\b|\bno(t)?\b.*\bwork(s)?\b',
                                           "Product functionality is impaired"),
        (r'\bwrong (item|product|size)\b', "Incorrect item received"),
    ]

    def gather(self, ctx: CustomerContext) -> CustomerContext:
        msg_lower = ctx.raw_message.lower()

        # ── Order record lookup (EC-2: None if not found) ────────────────
        if ctx.detected_order_id:
            ctx.order_record = ORDER_DB.get(ctx.detected_order_id)  # may be None

        # ── Policy snippets ───────────────────────────────────────────────
        policy_keys = CATEGORY_POLICY_MAP.get(ctx.category, [])
        ctx.policy_snippets = [POLICY_STORE[k] for k in policy_keys if k in POLICY_STORE]

        # ── Fact extraction ───────────────────────────────────────────────
        facts: List[str] = []
        for pattern, description in self._FACT_PATTERNS:
            if re.search(pattern, msg_lower):
                facts.append(description)
        ctx.extracted_facts = facts if facts else ["No additional qualifying facts found"]

        return ctx


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 3  —  RESOLUTION PLANNER
# ══════════════════════════════════════════════════════════════════════════════

class ResolutionPlanner:
    """
    Simulates an LLM reasoning step.
    Applies a decision tree to choose the optimal resolution type and plan.
    """

    def plan(self, ctx: CustomerContext) -> CustomerContext:
        cat   = ctx.category
        order = ctx.order_record

        # ── Order Status ─────────────────────────────────────────────────
        if cat == Category.ORDER_STATUS:
            if not ctx.detected_order_id:                            # EC-3
                ctx.resolution_type     = ResolutionType.ASK_ORDER_ID
                ctx.resolution_plan     = ("No order ID was detected. Ask the customer "
                                           "to provide it before looking up status.")
                ctx.needs_clarification = True
                ctx.clarifying_questions= ["Please share your order ID (format ORD-XXXX)."]
            elif order is None:                                      # EC-2
                ctx.resolution_type     = ResolutionType.ASK_ORDER_ID
                ctx.resolution_plan     = (f"{ctx.detected_order_id} was not found in "
                                           "our system. Request verification from customer.")
                ctx.needs_clarification = True
                ctx.clarifying_questions= [f"I couldn't find {ctx.detected_order_id}. "
                                           "Could you double-check the order number?"]
            elif order["status"] == "shipped":
                ctx.resolution_type = ResolutionType.PROVIDE_TRACKING
                ctx.resolution_plan = (f"Order shipped via {order['carrier']} on "
                                       f"{order['shipped_date']}. "
                                       f"Tracking: {order['tracking']}. "
                                       f"ETA: {order['est_delivery']}.")
            elif order["status"] == "processing":
                ctx.resolution_type = ResolutionType.PROCESSING_UPDATE
                ctx.resolution_plan = (f"Order is in warehouse processing. "
                                       f"Expected dispatch by {order['est_delivery']}.")
            elif order["status"] == "delivered":
                ctx.resolution_type = ResolutionType.CONFIRM_DELIVERED
                ctx.resolution_plan = ("Records show delivered. Ask customer to check "
                                       "neighbours/safe-place; offer investigation.")
            else:
                ctx.resolution_type = ResolutionType.PROVIDE_INFO
                ctx.resolution_plan = f"Order status is '{order['status']}'. Inform customer."

        # ── Returns / Refunds ─────────────────────────────────────────────
        elif cat == Category.RETURNS_REFUNDS:
            ctx.resolution_type = ResolutionType.INITIATE_RETURN
            ctx.resolution_plan = ("Confirm 30-day return eligibility; "
                                   "email prepaid return label within 24 hours.")

        # ── Product Issue ─────────────────────────────────────────────────
        elif cat == Category.PRODUCT_ISSUE:
            ctx.resolution_type = ResolutionType.REPLACEMENT_OR_REFUND
            ctx.resolution_plan = ("Apologise; offer free replacement or full refund. "
                                   "Request photo evidence if not already provided.")

        # ── Cancellation ──────────────────────────────────────────────────
        elif cat == Category.CANCELLATION:
            if order and order["status"] == "processing":
                ctx.resolution_type = ResolutionType.CANNOT_CANCEL
                ctx.resolution_plan = ("Order in fulfilment — cannot cancel. "
                                       "Advise refuse-delivery for automatic refund.")
            else:
                ctx.resolution_type = ResolutionType.CANCEL_ORDER
                ctx.resolution_plan = "Cancel order and issue immediate full refund."

        # ── Payment ───────────────────────────────────────────────────────
        elif cat == Category.PAYMENT:
            ctx.resolution_type = ResolutionType.ESCALATE_BILLING
            ctx.resolution_plan = ("Billing disputes require specialist review. "
                                   "Escalate; assure 24-hour callback.")

        # ── Shipping Inquiry / General ────────────────────────────────────
        else:
            ctx.resolution_type = ResolutionType.PROVIDE_INFO
            ctx.resolution_plan = "Answer using relevant policy information."

        return ctx


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 4  —  REPLY DRAFTER
# ══════════════════════════════════════════════════════════════════════════════

class ReplyDrafter:
    """
    Simulates an LLM generation step.
    Assembles a personalised, empathetic customer reply from context.
    """

    _EMPATHY = {
        Sentiment.ANGRY:      ("I'm truly sorry to hear about your experience — "
                               "I completely understand how upsetting this is, "
                               "and I want to resolve this for you immediately. "),
        Sentiment.FRUSTRATED: ("I'm sorry for the trouble you've been having — "
                               "let me get this sorted out for you right away. "),
        Sentiment.NEUTRAL:    "Thanks for getting in touch! ",
        Sentiment.POSITIVE:   "Thanks so much for contacting us — happy to help! ",
    }

    def _greeting(self, name: Optional[str]) -> str:
        return f"Hi {name}," if name else "Hi there,"

    def _core_content(self, ctx: CustomerContext) -> str:
        rtype = ctx.resolution_type
        o     = ctx.order_record or {}
        oid   = ctx.detected_order_id or "your order"

        if rtype == ResolutionType.PROVIDE_TRACKING:
            return (
                f"Your order **{oid}** ({o.get('item','')}) was picked up by "
                f"**{o.get('carrier','')}** on {o.get('shipped_date','')}. "
                f"Tracking number: **{o.get('tracking','')}**. "
                f"Estimated delivery: **{o.get('est_delivery','')}**. "
                "You can track it live on the carrier's website."
            )
        if rtype == ResolutionType.PROCESSING_UPDATE:
            return (
                f"Your order **{oid}** ({o.get('item','')}) is currently being "
                "prepared in our warehouse and hasn't shipped yet. "
                f"You can expect dispatch by **{o.get('est_delivery','')}** — "
                "we'll email you a tracking link the moment it leaves."
            )
        if rtype == ResolutionType.CONFIRM_DELIVERED:
            return (
                f"Our records show **{oid}** was marked as delivered. "
                "Could you check with a neighbour, at your door, or with your "
                "building reception? If it's still missing, reply here and I'll "
                "open a missing-parcel investigation immediately."
            )
        if rtype == ResolutionType.INITIATE_RETURN:
            return (
                f"No problem at all — you're within our 30-day return window. "
                "I'll email you a **prepaid return label** within 24 hours. "
                "Once we receive the item, your refund will be processed within "
                "3-5 business days to your original payment method."
            )
        if rtype == ResolutionType.REPLACEMENT_OR_REFUND:
            return (
                "I'm so sorry you received a defective item — that's not "
                "acceptable at all. I'd like to offer you a choice: we can "
                "send a **free replacement** right away, or issue a **full refund** "
                "within 3-5 business days. Which would you prefer?"
            )
        if rtype == ResolutionType.CANNOT_CANCEL:
            return (
                f"Unfortunately **{oid}** has already entered our fulfilment process "
                "and we're unable to stop it at this stage. When it arrives, simply "
                "**refuse the delivery** — the carrier will return it to us and "
                "we'll issue a full refund within 3-5 business days."
            )
        if rtype == ResolutionType.CANCEL_ORDER:
            return (
                f"Done! I've cancelled **{oid}** and a full refund has been initiated. "
                "You should see it on your statement within 3-5 business days."
            )
        if rtype == ResolutionType.ESCALATE_BILLING:
            return (
                "Billing matters require a specialist to review your account securely. "
                "I've flagged your case as **high priority** and a member of our billing "
                "team will contact you within **24 hours**. I apologise for the wait."
            )
        if rtype == ResolutionType.ASK_ORDER_ID:
            return (
                "Could you share your **order ID**? It looks like `ORD-XXXX` and "
                "you'll find it in your confirmation email. Once I have it I can "
                "pull up the full status for you!"
            )
        # PROVIDE_INFO fallback
        if ctx.policy_snippets:
            return " ".join(ctx.policy_snippets[:2])
        return ("I'd be happy to help! Could you give me a little more detail "
                "about your query so I can point you in the right direction?")

    def draft(self, ctx: CustomerContext) -> CustomerContext:
        greeting   = self._greeting(ctx.detected_name)
        empathy    = self._EMPATHY.get(ctx.sentiment, self._EMPATHY[Sentiment.NEUTRAL])
        core       = self._core_content(ctx)

        clarify_block = ""
        if ctx.clarifying_questions:
            clarify_block = "\n\nCould I also quickly check: " + \
                            " ".join(ctx.clarifying_questions)

        sign_off = "\n\nWarm regards,\n**Alex | ShopEase Support Team** 💙"

        ctx.draft_reply = f"{greeting}\n\n{empathy}{core}{clarify_block}{sign_off}"
        return ctx


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 5  —  QA CHECKER
# ══════════════════════════════════════════════════════════════════════════════

class QAChecker:
    """
    Simulates an LLM self-critique / review step.
    Evaluates the draft against a rubric and auto-patches where possible.
    """

    # Rubric: (label, test_fn, auto_fix_fn or None)
    def _build_rubric(self, ctx: CustomerContext):
        oid   = ctx.detected_order_id
        draft = ctx.draft_reply or ""
        dl    = draft.lower()

        return [
            (
                "Missing greeting",
                lambda d: not (re.search(r'\bhi\b|\bhello\b', d, re.IGNORECASE)),
                lambda d: "Hi there,\n\n" + d,
            ),
            (
                f"Order ID {oid} not mentioned",
                lambda d, o=oid: bool(o) and o not in d,
                None,   # cannot auto-fix without rewriting; leave as warning
            ),
            (
                "No clear next step",
                lambda d: not re.search(
                    r"will|you'll|i'll|we'll|can|please|click|visit|reply|email|choose",
                    d, re.IGNORECASE),
                None,
            ),
            (
                "Missing professional sign-off",
                lambda d: "support team" not in d.lower(),
                lambda d: d + "\n\nWarm regards,\n**Alex | ShopEase Support Team** 💙",
            ),
            (
                "Reply too brief (< 25 words)",
                lambda d: len(d.split()) < 25,
                None,
            ),
        ]

    def check(self, ctx: CustomerContext) -> CustomerContext:
        draft  = ctx.draft_reply or ""
        rubric = self._build_rubric(ctx)
        issues: List[str] = []

        for label, test_fn, fix_fn in rubric:
            if test_fn(draft):
                if fix_fn:
                    draft = fix_fn(draft)   # auto-patch
                else:
                    issues.append(label)    # flag for human review

        ctx.qa_issues   = issues
        ctx.qa_passed   = len(issues) == 0
        ctx.final_reply = draft
        return ctx


# ══════════════════════════════════════════════════════════════════════════════
# LOGGER  —  structured pretty-printer
# ══════════════════════════════════════════════════════════════════════════════

class Logger:
    """Handles all console output with consistent formatting."""
    W = 68   # total width

    @staticmethod
    def _wrap(text: str, indent: int = 4) -> str:
        prefix = " " * indent
        return textwrap.fill(text, width=Logger.W - indent,
                             initial_indent=prefix, subsequent_indent=prefix)

    def header(self, title: str):
        print(f"\n{'╔' + '═'*(self.W-2) + '╗'}")
        print(f"║{title.center(self.W-2)}║")
        print(f"{'╚' + '═'*(self.W-2) + '╝'}\n")

    def section(self, title: str):
        print(f"\n  {'━'*(self.W-4)}")
        print(f"  {title}")
        print(f"  {'━'*(self.W-4)}")

    def field(self, key: str, value, indent: int = 4):
        prefix = " " * indent
        label  = f"{key:<22}"
        val    = str(value) if value is not None else "—"
        # wrap long values
        full   = f"{prefix}{label}: {val}"
        if len(full) > self.W:
            print(f"{prefix}{label}:")
            for line in textwrap.wrap(val, width=self.W - indent - 2):
                print(f"{' '*(indent+2)}{line}")
        else:
            print(full)

    def bullet(self, text: str, indent: int = 6):
        prefix = " " * indent + "• "
        lines  = textwrap.wrap(text, width=self.W - indent - 2)
        print(prefix + lines[0])
        for l in lines[1:]:
            print(" " * (indent + 2) + l)

    def stage_banner(self, num: str, label: str, timing: str = ""):
        t = f"  [{timing}]" if timing else ""
        print(f"\n  ▶  Stage {num}: {label}{t}")

    def message_box(self, label: str, text: str):
        bar = "─" * (self.W - 4)
        print(f"\n  {bar}")
        print(f"  {label}")
        print(f"  {bar}")
        plain = text.replace("**", "").replace("`", "")
        for line in plain.split("\n"):
            print(f"    {line}")
        print(f"  {bar}")

    def status(self, ok: bool, text: str, indent: int = 4):
        icon = "✅" if ok else "⚠️ "
        print(f"{' '*indent}{icon}  {text}")

    def divider(self):
        print(f"\n  {'═'*(self.W-4)}")

    def log_stage1(self, ctx: CustomerContext):
        self.stage_banner("1", "Classify")
        cat = ctx.category.value if hasattr(ctx.category, 'value') else ctx.category
        sub = ctx.subcategory
        sen = ctx.sentiment.value if hasattr(ctx.sentiment, 'value') else ctx.sentiment
        pri = ctx.priority.value if hasattr(ctx.priority, 'value') else ctx.priority
        self.field("Category",   cat)
        self.field("Subcategory",sub)
        self.field("Sentiment",  sen)
        self.field("Priority",   pri)
        self.field("Order ID",   ctx.detected_order_id)
        self.field("Customer",   ctx.detected_name)

    def log_stage2(self, ctx: CustomerContext):
        self.stage_banner("2", "Gather Context")
        status = ctx.order_record["status"] if ctx.order_record else "not found"
        self.field("Order status",   status)
        self.field("Policies found", len(ctx.policy_snippets))
        self.field("Facts",          len(ctx.extracted_facts))
        for f in ctx.extracted_facts:
            self.bullet(f)

    def log_stage3(self, ctx: CustomerContext):
        self.stage_banner("3", "Determine Resolution")
        rtype = ctx.resolution_type.value if hasattr(ctx.resolution_type, 'value') else ctx.resolution_type
        self.field("Resolution type", rtype)
        self.field("Plan", ctx.resolution_plan)
        if ctx.clarifying_questions:
            self.field("Clarify?", "YES")
            for q in ctx.clarifying_questions:
                self.bullet(q)

    def log_stage4(self, ctx: CustomerContext):
        self.stage_banner("4", "Draft Reply")
        words = len((ctx.draft_reply or "").split())
        self.field("Draft word count", words)

    def log_stage5(self, ctx: CustomerContext):
        self.stage_banner("5", "QA Check")
        self.status(ctx.qa_passed, "QA PASSED" if ctx.qa_passed
                    else f"QA FLAGGED — {len(ctx.qa_issues)} issue(s)")
        for issue in ctx.qa_issues:
            self.bullet(f"⚠  {issue}")


# ══════════════════════════════════════════════════════════════════════════════
# CHAIN ORCHESTRATOR
# ══════════════════════════════════════════════════════════════════════════════

class Chain:
    """
    Wires Classifier → ContextGatherer → ResolutionPlanner
             → ReplyDrafter → QAChecker into a sequential pipeline.
    Each stage wraps execution in error handling so one failure
    doesn't crash the entire chain.
    """

    def __init__(self):
        self._classifier  = Classifier()
        self._gatherer    = ContextGatherer()
        self._planner     = ResolutionPlanner()
        self._drafter     = ReplyDrafter()
        self._qa          = QAChecker()
        self._log         = Logger()

    def _run_stage(self, num: str, fn, ctx: CustomerContext,
                   log_fn=None) -> CustomerContext:
        try:
            ctx = fn(ctx)
        except Exception as exc:                    # catch unexpected errors
            ctx.errors.append(f"Stage {num}: {exc}")
            ctx.final_reply = ("I'm sorry, we encountered a technical issue. "
                               "A support specialist will contact you shortly.")
        if log_fn:
            log_fn(ctx)
        return ctx

    def run(self, message: str, verbose: bool = True) -> CustomerContext:
        """
        Execute the full 5-stage chain.

        Args:
            message : raw customer message (EC-1 validated in __post_init__)
            verbose : print stage-by-stage log

        Returns:
            Completed CustomerContext with final_reply populated.
        """
        message = str(message)                        # EC-7: coerce before anything else
        ctx = CustomerContext(raw_message=message)   # EC-1 validated here
        log = self._log

        if verbose:
            log.section(f"CUSTOMER MESSAGE")
            for line in textwrap.wrap(message, width=60):
                print(f"    {line}")

        stages = [
            ("1", self._classifier.classify,  log.log_stage1),
            ("2", self._gatherer.gather,       log.log_stage2),
            ("3", self._planner.plan,          log.log_stage3),
            ("4", self._drafter.draft,         log.log_stage4),
            ("5", self._qa.check,              log.log_stage5),
        ]

        for num, fn, log_fn in stages:
            ctx = self._run_stage(num, fn, ctx, log_fn if verbose else None)
            if ctx.errors:
                if verbose:
                    log.status(False, f"Error in Stage {num} — aborting chain.")
                break

        if verbose:
            log.message_box("📬  FINAL SUPPORT REPLY", ctx.final_reply or "")

        return ctx


# ══════════════════════════════════════════════════════════════════════════════
# DEMO & INTERACTIVE MODES
# ══════════════════════════════════════════════════════════════════════════════

DEMO_CASES = [
    {
        "title": "😤 Frustrated — late delivery (order exists, shipped)",
        "message": (
            "Hi, I'm still waiting for my headphones (ORD-1001) and they were "
            "supposed to arrive last week. I've been waiting for so long — "
            "this is really frustrating. Where is my order??"
        ),
    },
    {
        "title": "📦 Return request (delivered order)",
        "message": (
            "Hello, my name is Maria. I received my 4K Webcam (ORD-1003) "
            "but it's not compatible with my setup. I'd like to return it "
            "and get a refund please."
        ),
    },
    {
        "title": "😡 Angry customer — broken product",
        "message": (
            "This is absolutely unacceptable! The keyboard I received (ORD-1002) "
            "is completely defective — half the keys don't work at all. "
            "I want a replacement NOW."
        ),
    },
    {
        "title": "🚚 Shipping policy question (no order ID)",
        "message": (
            "Hi, do you offer free shipping? How much does express delivery cost?"
        ),
    },
    {
        "title": "❌ Cancellation — order already in processing",
        "message": (
            "Hi, I just placed an order (ORD-1002) by mistake and need to "
            "cancel it immediately. Can you cancel it for me?"
        ),
    },
]

# ── Edge-case test messages (Stage 4: Fix and Iterate validation) ─────────────
EDGE_CASES = [
    {
        "title": "EC-1: Empty message (should raise ValueError)",
        "message": "   ",   # whitespace-only
        "expect_error": True,
    },
    {
        "title": "EC-2: Order ID not in DB",
        "message": "Where is my order ORD-9999?",
        "expect_error": False,
    },
    {
        "title": "EC-3: No order ID detected",
        "message": "Where is my order? I can't find the tracking number.",
        "expect_error": False,
    },
    {
        "title": "EC-7: Non-string input (integer)",
        "message": 12345,
        "expect_error": False,
    },
]


def run_demo():
    chain = Chain()
    log   = Logger()

    log.header("ReACT Prompt Chain — Customer Support AI  |  DEMO MODE")
    print("  5-Stage Pipeline: Classify → Context → Resolve → Draft → QA\n")

    for i, case in enumerate(DEMO_CASES, 1):
        log.divider()
        print(f"\n  CASE {i}: {case['title']}")
        chain.run(case["message"], verbose=True)
        input("\n  ↵  Press Enter for the next case...")

    log.divider()
    print("\n  ✅  All demo cases complete.\n")


def run_edge_cases():
    chain = Chain()
    log   = Logger()

    log.header("ReACT Stage 4 — Edge Case Verification")

    for i, case in enumerate(EDGE_CASES, 1):
        log.divider()
        print(f"\n  EDGE CASE {i}: {case['title']}")
        if case.get("expect_error"):
            try:
                chain.run(str(case["message"]).strip() or "", verbose=False)
                log.status(False, "Expected ValueError — NOT raised ❌")
            except ValueError as e:
                log.status(True, f"ValueError raised correctly: {e}")
        else:
            ctx = chain.run(case["message"], verbose=True)
            ok  = bool(ctx.final_reply)
            log.status(ok, "Chain completed with final reply" if ok
                       else "Chain produced no reply")

    print()


def run_interactive():
    chain = Chain()
    log   = Logger()

    log.header("ReACT Prompt Chain — Interactive Mode")
    print("  Type a customer support message, or 'quit' to exit.\n")
    print("  Sample order IDs: ORD-1001 (shipped), ORD-1002 (processing),")
    print("                    ORD-1003 (delivered), ORD-1004 (cancelled)\n")

    while True:
        try:
            msg = input("  You ▶  ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n  Goodbye! 👋")
            break

        if not msg:
            continue
        if msg.lower() in {"quit", "exit", "q"}:
            print("\n  Goodbye! 👋")
            break

        try:
            chain.run(msg, verbose=True)
        except ValueError as e:
            log.status(False, f"Input error: {e}")

        print()


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

def main():
    """
    Modes:
      python solution.py              → demo (5 representative cases)
      python solution.py --edges      → edge-case verification
      python solution.py --interactive→ REPL mode
    """
    args = sys.argv[1:]
    if "--interactive" in args:
        run_interactive()
    elif "--edges" in args:
        run_edge_cases()
    else:
        run_demo()


if __name__ == "__main__":
    main()


"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STAGE 3: EXECUTION AND OBSERVATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Representative test run — Case 3: Angry customer / broken product

EXPECTED STDOUT LOG:
───────────────────
  ▶  Stage 1: Classify
      category          : product_issue
      sentiment         : angry
      priority          : high        (product_issue override applied)
      order_id          : ORD-1002

  ▶  Stage 2: Gather Context
      order_status      : processing
      policies_found    : 3
      facts             : ["Product functionality is impaired"]

  ▶  Stage 3: Determine Resolution
      resolution_type   : send_replacement_or_refund
      plan              : Apologise; offer free replacement or full refund.

  ▶  Stage 4: Draft Reply
      draft_word_count  : ~80 words

  ▶  Stage 5: QA Check
      qa_passed         : True   (greeting ✓, sign-off ✓, next-step ✓)

  📬  FINAL REPLY — starts with empathy opener for ANGRY sentiment,
      offers replacement OR refund choice, signed by Alex.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STAGE 4: FIX AND ITERATE  —  REVISION NOTES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Iteration 1 — Initial design
  • Flat script with one classify() function; no state object.
  • Problem: data passed as loose variables → hard to extend.

Iteration 2 — Introduced CustomerContext dataclass
  • Centralised state; each stage reads/writes fields.
  • Problem: classification priority incorrect — "deliver" keyword
    matched ORDER_STATUS before SHIPPING_INQUIRY for policy queries.

Iteration 3 — Fixed classifier rule ordering
  • High-specificity categories (cancellation, product_issue) checked first.
  • SHIPPING_INQUIRY checked before ORDER_STATUS to avoid "delivery" clash.
  • Problem: order ID regex produced "ORD--1001" (double dash).

Iteration 4 — Fixed order ID regex normalisation
  • Extract digits only, then prepend "ORD-" → always produces "ORD-XXXX".
  • Problem: EC-2 (unknown order ID) caused KeyError in policy lookup.

Iteration 5 — Added EC-2 guard in ContextGatherer
  • ORDER_DB.get() returns None safely; ResolutionPlanner checks for None.
  • Problem: QA checker raised exception when draft_reply was None
    (chain error path).

Iteration 6 — Final hardening
  • Chain._run_stage() wraps each stage in try/except.
  • QAChecker guards against None draft_reply.
  • CustomerContext.__post_init__ validates and coerces input (EC-1, EC-7).
  • Added enum classes for all string constants → prevents typo bugs.
  • Separated Logger class → all display logic in one place.
  • Added --edges CLI flag for edge-case regression suite.

Result: All 5 demo cases + 4 edge cases pass cleanly.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""


╔══════════════════════════════════════════════════════════════════╗
║      ReACT Prompt Chain — Customer Support AI  |  DEMO MODE      ║
╚══════════════════════════════════════════════════════════════════╝

  5-Stage Pipeline: Classify → Context → Resolve → Draft → QA


  ════════════════════════════════════════════════════════════════

  CASE 1: 😤 Frustrated — late delivery (order exists, shipped)

  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CUSTOMER MESSAGE
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Hi, I'm still waiting for my headphones (ORD-1001) and they
    were supposed to arrive last week. I've been waiting for so
    long — this is really frustrating. Where is my order??

  ▶  Stage 1: Classify
    Category              : order_status
    Subcategory           : order status
    Sentiment             : neutral
    Priority              : medium
    Order ID              : ORD-1001
    Customer              : —

  ▶  Stage

'\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\nSTAGE 3: EXECUTION AND OBSERVATION\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\nRepresentative test run — Case 3: Angry customer / broken product\n\nEXPECTED STDOUT LOG:\n───────────────────\n  ▶  Stage 1: Classify\n      category          : product_issue\n      sentiment         : angry\n      priority          : high        (product_issue override applied)\n      order_id          : ORD-1002\n\n  ▶  Stage 2: Gather Context\n      order_status      : processing\n      policies_found    : 3\n      facts             : ["Product functionality is impaired"]\n\n  ▶  Stage 3: Determine Resolution\n      resolution_type   : send_replacement_or_refund\n      plan              : Apologise; offer free replacement or full refund.\n\n  ▶  Stage 4: Draft Reply\n      draft_word_count  : ~80 words\n\n  ▶  Stage 5: QA Check\n      qa_passed         : True   (greeting ✓, sign-off ✓, ne